# Code 11.2.1: Sampling responses and their log-probabilities

`generate` samples one response per prompt from the policy at temperature 1 with no truncation, and returns a mask that is 1 on response tokens up to and including the first EOS. `response_logprobs` computes $\log \pi(y_t \mid x, y_{\lt t})$ for every response token with one forward pass. Prompts are left-padded so that all responses start at the same position; the position ids are computed from the attention mask so that padding does not shift positions. GPT-2 has no padding token, so EOS doubles as padding, and the mask, not the token id, marks which tokens belong to the response. Section 7 reuses both functions.


In [ ]:
import torch
import torch.nn.functional as F

def generate(model, tok, prompts, max_new_tokens=24, temperature=1.0):
    """Sample one response per prompt. Returns token ids, an attention mask,
    and a mask that is 1 on response tokens up to and including the first EOS."""
    enc = tok(prompts, return_tensors="pt", padding=True)      # left-padded
    with torch.no_grad():
        seqs = model.generate(**enc, do_sample=True, top_k=0, top_p=1.0,
                              temperature=temperature, max_new_tokens=max_new_tokens,
                              pad_token_id=tok.pad_token_id)
    P = enc["input_ids"].shape[1]                              # prompt length (with padding)
    resp = seqs[:, P:]
    is_eos = resp == tok.eos_token_id
    after_eos = (is_eos.cumsum(dim=1) - is_eos.long()) > 0     # strictly after the first EOS
    resp_mask = (~after_eos).long()
    attn = torch.cat([enc["attention_mask"], resp_mask], dim=1)
    return seqs, attn, resp_mask, P

def response_logprobs(model, seqs, attn, P):
    """log pi(y_t | x, y_<t) for each response position t: shape (B, T)."""
    pos = (attn.cumsum(dim=1) - 1).clamp(min=0)                # correct positions under left padding
    logits = model(seqs, attention_mask=attn, position_ids=pos).logits
    logits = logits[:, P - 1:-1].float()                       # the logits that predict y_1 .. y_T
    logp = F.log_softmax(logits, dim=-1)
    return logp.gather(-1, seqs[:, P:].unsqueeze(-1)).squeeze(-1)
